In [3]:
!pip install dotenv

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)

   ---------------------------------------- 2/2 [dotenv]



In [58]:
# /db/dependencies.py
from dataclasses import dataclass
from typing import Any

@dataclass
class GraphDependencies:
    # In a real scenario, this would be your ArcadeDB client connection
    db_connection: Any 
    user_id: str

# /schemas/graph_schemas.py
from pydantic import BaseModel, Field

class CypherQuery(BaseModel):
    query: str = Field(..., description="The exact Cypher query to execute in ArcadeDB.")
    rationale: str = Field(..., description="Why this query is necessary based on the user's prompt.")

In [72]:
# /skills/graph_capability.py

from pydantic_ai import RunContext



from pydantic_ai.capabilities import Capability

memory = Capability(id="GraphMemory",
                    description="Grants access to the user's graph memory for storing and retrieving information.",
                    instructions= (
            "You have direct access to the user's ArcadeDB graph memory. "
            "CRITICAL RULE: Before attempting to store new data or create a new node, "
            "you must query the database to check if a relevant schema already exists."
        ),
                    defer_loading=False)

@memory.tool
async def execute_graph_query(ctx: RunContext[GraphDependencies], query_data: CypherQuery) -> str:
    """
    Executes a Cypher query against the user's isolated memory graph.
    """
    db = ctx.deps.db_connection
    user_id = ctx.deps.user_id

    # Execution Layer Safety Check
    if "DROP" in query_data.query.upper() or "DELETE" in query_data.query.upper():
        return "Error: Destructive queries are not permitted."

    try:
        # result = await db.query(query_data.query)
        result = f"Successfully executed {query_data.query} for user {user_id}"
        return result
    except Exception as e:
        return f"Database error: {str(e)}"


In [73]:
from pydantic_ai import Agent, ThinkingPart
from pydantic_ai.models.ollama import OllamaModel
from pydantic_ai.capabilities import Thinking
from dotenv import load_dotenv
load_dotenv()

model = OllamaModel('google/gemma-4-26b-a4b-qat')
agent = Agent(model,deps_type=GraphDependencies,
              capabilities=[Thinking(effort='minimal'), memory])


In [74]:
from pydantic_ai import ThinkingPartDelta, TextPart, TextPartDelta, PartDeltaEvent,PartStartEvent
dependencies=GraphDependencies(db_connection="http://localhost:2480", user_id="default")
async for event in agent.run_stream_events("Please get me a property from the db. check the schema first but i want a two bedroom apartment in recoleta", deps=dependencies):
    if isinstance(event, PartStartEvent):
        print(f"\n--- {event.part.id} started ---\n")
        event = event.part
    elif isinstance(event, PartDeltaEvent):
        event = event.delta
    if isinstance(event, ThinkingPartDelta):
        print(f"\033[94m{event.content_delta}\033[0m",end="")
    elif isinstance(event, TextPart):
        print(event.content,end="")
    elif isinstance(event, TextPartDelta):
        print(event.content_delta,end="")
    else:
        print(event)

C:\Users\ianda\AppData\Local\Temp\ipykernel_47016\3123997101.py:3: DeprecationWarning: Iterating `AgentEventStream` directly with `async for event in stream:` is deprecated. Use `async with agent.run_stream_events(...) as stream:` and `async for event in stream:` instead to ensure proper cleanup.
  async for event in agent.run_stream_events("Please get me a property from the db. check the schema first but i want a two bedroom apartment in recoleta", deps=dependencies):



--- reasoning_content started ---

ThinkingPart(content='\n', id='reasoning_content', provider_name='ollama')
The user wants to find a "two bedroom apartment in recoleta" from the database.
I must follow the critical rule: "Before attempting to store new data or create a new node, you must query the database to check if a relevant schema already exists."
However, the user is asking to *get* a property, not store one. But to get it correctly, I need to know the schema (node labels and property names).

First, I should explore the schema to understand how "apartment", "recoleta" (location), and "two bedrooms" (number of bedrooms) are represented.

Steps:
1. Query the schema to find relevant labels and properties.
2. Based on the schema, construct a Cypher query to find the requested property.

Let's start by checking the schema. I'll use `CALL db.schema.visualization()` or just list labels and properties. Since I can't "visualize", I'll use `CALL db.schema.nodes()` and `CALL db.schema.r

In [ ]:
async def execute_graph_query(ctx: RunContext[GraphDependencies], query_data: CypherQuery) -> str:
    """
    Executes a Cypher query against the user's isolated memory graph.
    """
    db = ctx.deps.db_connection
    user_id = ctx.deps.user_id

    # Execution Layer Safety Check
    if "DROP" in query_data.query.upper() or "DELETE" in query_data.query.upper():
        return "Error: Destructive queries are not permitted."

    try:
        result = await db.query(query_data.query)
        result = f"Successfully executed {query_data.query} for user {user_id}"
        return result
    except Exception as e:
        return f"Database error: {str(e)}"